# Session 5: Tidy data and the sample table

The Fifth Unregulated Contaminant Monitoring Rule (UCMR 5) had every public water system in the United States serving more than 3,300 people, and a sample of the smaller ones, measure 29 PFAS in their drinking water between 2023 and 2025, and the EPA publishes every result. `data/states/` holds, for each state, the first sampling event's results for six of those PFAS in the shape a laboratory report takes, one row per sample and one column per compound, and a sample table with one row per sample point. Section 1 takes Washington's report apart against the three rules of tidy data, puts it back together as one measurement per row, joins the sample table, and computes detection rates. Section 2 does the same for a state assigned to you. Section 3 asks what one row stands for, and shows that the detection rate depends on the answer. Section 4 is the judgment cell.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 5 assignment on Canvas by 1:30 pm on Tuesday, October 20. Submissions are individual. Every state's files are in the repository, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, NumPy, pandas and Matplotlib. pandas is new this session: it is the library for tables, and its `DataFrame` is a table with named columns that `read_csv` builds from a file. The cell prints nothing when it works.

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameters

The folder of state files, Washington's two files for section 1 and the state code for section 2, the six compounds in the order the tables list them, the prefix the laboratory used for a result below its reporting limit, and the maximum contaminant level for PFOA that the EPA set in April 2024, which happens to equal the reporting limit in this program. Leave `STATE` as `CHANGE-ME` until section 2.

In [ ]:
STATES_DIR = "data/states/"   # two files per state, <ST>_results.csv and <ST>_samples.csv
WA_RESULTS = STATES_DIR + "WA_results.csv"
WA_SAMPLES = STATES_DIR + "WA_samples.csv"
STATE = "CHANGE-ME"   # your assigned state, two letters, section 2
PFAS = ["PFBS", "PFHxA", "PFHxS", "PFOA", "PFOS", "PFPeA"]
BDL_PREFIX = "<"   # a result below the reporting limit reads "< 0.004"
MCL_PFOA = 0.004   # ug/L; the EPA maximum contaminant level for PFOA, April 2024

## 1. Explore: a laboratory report read against the three rules

To see why a table that is easy to read can be hard to use, load Washington's report and look at its first rows. `read_csv` returns a `DataFrame`; `.shape` is its row and column count, and `.head()` its first five rows.

In [ ]:
results = pd.read_csv(WA_RESULTS)
print(results.shape)
results.head()

Each row is one sample, one column per compound, and a reader can scan a row for all six results. Against the three rules of tidy data it breaks all three: the compound is a column name rather than a value in a column (one variable per column); a row bundles six measurements that a question about PFOA alone has to pick apart (one observation per row); and two kinds of cell hold more than one value, `sample_id` with four facts joined by underscores, and `< 0.004`, which is a code for "below the reporting limit of 0.004" (one value per cell). Section 1 repairs the three in four steps. The first splits `sample_id`: `.str.split("_", expand=True)` applies session 4's `split` to every row and returns one column per piece.

In [ ]:
parts = results["sample_id"].str.split("_", expand=True)
results["pwsid"] = parts[0]
results["sample_point_id"] = parts[1]
results["collection_date"] = parts[2]
results["event"] = parts[3]
results[["sample_id", "pwsid", "sample_point_id", "collection_date", "event"]].head(3)

The second step turns each compound column into numbers. A result below the reporting limit is not zero and not a number the laboratory measured; it is a measurement that was made and did not reach the limit, so it becomes `NaN`, pandas' marker for a missing value, which `mean` and `count` skip. The loop reads each column as text with `astype(str)`, marks the rows that start with `<`, sets them to `NaN` with `.loc`, and converts the rest with `astype(float)`; reading as text first is what lets the cell run a second time, after the column is already numeric. The printout is the count of below-limit results per compound two ways, from the `<` marks and from `.isna().sum()` afterwards, which agree because every cell in these files is a value or a below-limit code.

In [ ]:
below_limit_count = {}
for compound in PFAS:
    text = results[compound].astype(str)
    below_limit = text.str.startswith(BDL_PREFIX)
    below_limit_count[compound] = int(below_limit.sum())
    results.loc[below_limit, compound] = np.nan   # below the reporting limit: missing, not zero
    results[compound] = results[compound].astype(float)

print("below-limit codes:", below_limit_count)
print("NaN after conversion:", results[PFAS].isna().sum().to_dict())

The third step reshapes the table so that each row is one measurement: `melt` keeps the identifier columns (`id_vars`), turns the six compound columns (`value_vars`) into two, one holding the old column name (`var_name`) and one the value (`value_name`). Six times as many rows, and `PFOA` is now a value you can filter on. A `detected` column, true where the concentration is not `NaN`, records the laboratory's yes or no.

In [ ]:
tidy = results.melt(id_vars=["sample_id", "pwsid", "sample_point_id", "collection_date", "event"],
                    value_vars=PFAS, var_name="compound", value_name="concentration_ug_L")
tidy["detected"] = ~tidy["concentration_ug_L"].isna()
print(tidy.shape)
tidy.head()

The fourth step joins the sample table, which records what each sample point is: the system it belongs to, the system's name and size, and whether the facility draws groundwater or surface water. The identifier columns are read as text with `dtype`, because a sample point id such as `001` is a label, and `read_csv` would otherwise turn it into the number 1 and lose the zeros, a value changed by the reading. `merge` joins on the two columns that identify a sample point, and `how="left"` keeps every measurement row whether or not the sample table has a match, so a missing match shows as `NaN` instead of a vanished row. The printout is the row count before and after, which a left merge leaves equal, and the number of rows an inner merge would have dropped.

In [ ]:
samples = pd.read_csv(WA_SAMPLES, dtype={"pwsid": str, "sample_point_id": str})
merged = tidy.merge(samples, on=["pwsid", "sample_point_id"], how="left")
inner = tidy.merge(samples, on=["pwsid", "sample_point_id"], how="inner")
print(len(tidy), "rows before the merge,", len(merged), "after a left merge,", len(tidy) - len(inner), "rows an inner merge would drop")
print(merged["facility_water_type"].isna().sum(), "measurements with no sample table match")

With one measurement per row and the water type on every row, a detection rate is one `groupby`: group the rows by compound, take the mean of `detected`, which is the fraction of true values. Grouping by compound and water type at once gives the rate for groundwater (GW) and surface water (SW) facilities side by side.

In [ ]:
rate_by_compound = merged.groupby("compound")["detected"].mean()
print(rate_by_compound.round(3))
rate_by_type = merged.groupby(["compound", "facility_water_type"])["detected"].mean()
print(rate_by_type.round(3))

The plot shows the detection rate per compound as a bar, in percent. One series, so no legend.

In [ ]:
fig, ax = plt.subplots()
ax.bar(rate_by_compound.index, 100 * rate_by_compound.values, color="black")
ax.set_xlabel("compound")
ax.set_ylabel("results at or above the reporting limit (%)")
ax.set_title("Washington, UCMR 5 first sampling event")
fig.tight_layout()
plt.show()

Six bars, all under 10%: PFBS is the most detected of the six in Washington at 8.0% of results, then PFOS at 5.5%, PFHxS at 4.3%, PFPeA at 4.1%, PFHxA at 3.1%, and PFOA last at 2.0%; every detection is in a groundwater (GW) or groundwater-under-influence (GU) facility, and the surface water (SW) and mixed (MX) rates are zero for all six. The next cell is the check that decides how a below-limit result is encoded: the mean PFOA concentration over the detected results, and the mean the table would report if every below-limit result were zero. The first is a mean of measurements; the second is pulled toward zero by results that were never measured as zero.

In [ ]:
pfoa = merged[merged["compound"] == "PFOA"]["concentration_ug_L"]
print(f"PFOA results {pfoa.size}, detected {pfoa.count()}, mean of the detected {pfoa.mean():.5f} ug/L")
print(f"mean if the below-limit results were zero: {pfoa.fillna(0).mean():.5f} ug/L, against the MCL of {MCL_PFOA} ug/L")

**Explain.** The four steps were: split the identifier, below-limit to `NaN`, melt, merge. In two sentences, say which of the three rules each of the first three steps restores, and what the fourth step adds that no rule covers.

*Your answer:*

## 2. Apply: your assigned state

Each of you has one state from the list on the board. Set `STATE` in Parameters to its two-letter code; the files `<STATE>_results.csv` and `<STATE>_samples.csv` are already in the repository's `data/states/` folder, so there is nothing to download.

**[TASK 1]** Set `STATE` and run the next cell, which loads your state's two tables and prints their shapes. Until `STATE` is set, the cell stops with `FileNotFoundError`, which is what Run All should do until the task is done.

In [ ]:
my_results = pd.read_csv(STATES_DIR + STATE + "_results.csv")
my_samples = pd.read_csv(STATES_DIR + STATE + "_samples.csv", dtype={"pwsid": str, "sample_point_id": str})
print(STATE, "results", my_results.shape, "sample table", my_samples.shape)

**Ask the assistant** before the next task, in Claude Code, with a specific prompt, today's AI practice. `My PFAS columns hold strings like "< 0.004". How do I make them numeric?` gets code that runs. A prompt that names the table, the columns, the constraint and what done looks like gets code you can check, so type this one, with your state's code in the file name:

`In my notebook, my_results is data/states/<STATE>_results.csv. Its six PFAS columns hold numbers as text, and strings that start with "<" for a result below the reporting limit. Show me how to make those columns numeric so that a below-limit result becomes NaN, and how to check that the number of NaN values in each column equals the number of "<" strings before.`

Read its answer against section 1 and against this notebook: whether a below-limit result becomes `NaN`, not 0, and not silently, whether the check it proposes would pass for your state, and whether its code uses the names already in Parameters. What it suggested and what you did with it is one candidate for part (b) of the judgment cell.

**[TASK 2]** Section 1's first three steps as one function, so that any state's report becomes tidy in one call. The function below is section 1's three cells with `wide` in place of `results`, and one line left to you: what a below-limit result becomes. Section 1 made a choice there; make it again here, in your own function, and TASK 4 shows its consequence. Until the line is filled, the call below the definition stops with a `TypeError` at that line, because pandas cannot assign the `...` placeholder.

In [ ]:
def tidy_results(wide):
    """Tidy a wide laboratory report: one measurement per row, below-limit results as NaN.

    Parameters
    ----------
    wide : pandas.DataFrame
        One row per sample: sample_id, then one column per compound in PFAS, values as
        reported ("0.0040") or "< MRL". Left unchanged; the function works on a copy.

    Returns
    -------
    pandas.DataFrame
        One row per measurement: the identifier columns, compound, concentration_ug_L
        (NaN below the reporting limit) and detected.
    """
    wide = wide.copy()   # work on a copy, so the table passed in is unchanged and the cell can run again
    parts = wide["sample_id"].str.split("_", expand=True)
    wide["pwsid"] = parts[0]
    wide["sample_point_id"] = parts[1]
    wide["collection_date"] = parts[2]
    wide["event"] = parts[3]
    for compound in PFAS:
        text = wide[compound].astype(str)
        below_limit = text.str.startswith(BDL_PREFIX)
        wide.loc[below_limit, compound] = ...   # [TASK 2] what a below-limit result becomes
        wide[compound] = wide[compound].astype(float)
    tidy_table = wide.melt(id_vars=["sample_id", "pwsid", "sample_point_id", "collection_date", "event"],
                           value_vars=PFAS, var_name="compound", value_name="concentration_ug_L")
    tidy_table["detected"] = ~tidy_table["concentration_ug_L"].isna()
    return tidy_table


my_tidy = tidy_results(my_results)
print(my_tidy.shape)
print(my_tidy.groupby("compound")["detected"].mean().round(3))

**Ask the assistant** again before the next task: `What is the difference between how="left" and how="inner" in pandas merge, and which should I use to attach a sample table to measurements?`. Read its answer against the row counts the cell prints.

**[TASK 3]** Join your tidy table to your sample table. The stub leaves `how=` to you; until it is filled, the cell stops with `ValueError: 'Ellipsis' is not a valid Merge type`. The printout is the row count before and after, and the rows an inner merge would drop.

In [ ]:
my_merged = my_tidy.merge(my_samples, on=["pwsid", "sample_point_id"], how=...)   # [TASK 3] left or inner
my_inner = my_tidy.merge(my_samples, on=["pwsid", "sample_point_id"], how="inner")
print(len(my_tidy), "rows before,", len(my_merged), "after,", len(my_tidy) - len(my_inner), "rows an inner merge would drop")

**[TASK 4]** PFOA in your state by facility water type: the detection rate and the mean detected concentration. The stub filters the PFOA rows and leaves the `groupby` column to you; until it is filled, the cell stops with a `TypeError` about an ellipsis. The last line is the zero-substitution mean, for part (a).

In [ ]:
my_pfoa = my_merged[my_merged["compound"] == "PFOA"]
by_type = my_pfoa.groupby(...)   # [TASK 4] the sample table column that says groundwater or surface water
print("results per water type:   ", by_type["detected"].size().to_dict())
print("detections per water type:", by_type["detected"].sum().to_dict())
print(by_type["detected"].mean().round(3))
print(by_type["concentration_ug_L"].mean().round(5))
print(f"all facilities: detected in {my_pfoa['detected'].sum()} of {len(my_pfoa)} results, mean of the detected {my_pfoa['concentration_ug_L'].mean():.5f} ug/L")
print(f"mean if the below-limit results were zero: {my_pfoa['concentration_ug_L'].fillna(0).mean():.5f} ug/L")

**Explain.** Compare with a neighbor: give both states' PFOA detection rates and mean detected concentrations, and say which state's numbers are the more certain, by the count of results behind each.

*Your answer:*

## 3. Analyze: what counts as an observation

A detection rate is a count of detections over a count of something, and the something is a choice. Per result, it is the fraction of PFOA measurements above the reporting limit, which is what the laboratory's table gives directly. Per sample point, it is the fraction of entry points with at least one detection. Per system, it is the fraction of water systems with at least one detection. The next cell computes all three for your state with `groupby` and `.any()`, which is true for a group if any row in it is true, and prints each with its denominator.

In [ ]:
per_result = my_pfoa["detected"].mean()
by_point = my_pfoa.groupby(["pwsid", "sample_point_id"])["detected"].any()
by_system = my_pfoa.groupby("pwsid")["detected"].any()
print(f"per result:       {my_pfoa['detected'].sum():4d} of {len(my_pfoa):4d} results       {100 * per_result:5.1f}%")
print(f"per sample point: {by_point.sum():4d} of {by_point.size:4d} sample points {100 * by_point.mean():5.1f}%")
print(f"per system:       {by_system.sum():4d} of {by_system.size:4d} systems       {100 * by_system.mean():5.1f}%")

**Explain.** Which of the three rates answers "how many of this state's water systems delivered PFOA at or above the MCL in the first sampling event", and why do the other two not? Two of your three rates agree to within a sample: say why, and what a sample point would have to do for them to differ. Use your numbers. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): report your state's PFOA detection rate per result and per system, each with its denominator, and the mean detected PFOA concentration; say which rate answers the systems question and why; and say what the mean would have been had the below-limit results been zero, from the last line of TASK 4. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your tables. The two prompts in section 2 are the natural sources; the reason has to cite a count or a value from your state, not a general rule. This session's AI practice, named in the notes, is *specific prompts*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__}")